# Certainty Evidence Tree: End-to-End Demo

## What is an Evidence Tree?

An **evidence tree** is a hierarchical proof structure that explains *why* a candidate
was derived. Each node has a `node_kind` from the provenance-role taxonomy:

```
candidate_result          ← the derived fact (root)
├── support_section       ← what evidence supports it?
│   ├── non_fact_check    ← structural checks (ruleref, eq, etc.)
│   └── non_fact_check
└── rule_ref_section      ← which child rules were invoked?
    └── rule_ref          ← pointer to child rule
        └── referenced_support    ← the child proof subtree
            └── support_section
                ├── predicate_witness_group  ← matching facts for a predicate
                │   └── assertion_fact       ← the actual witness fact
                └── predicate_witness_group
                    └── assertion_fact
```

## What is Certainty Propagation?

When a rule declares `condition_weights`, certainty measures how well each
condition in the child proof is satisfied:

- Each **condition** (predicate witness group) gets a `weight` from the rule metadata
- Each condition's **impact** = `weight × confidence` (confidence defaults to 1.0)
- The **aggregate certainty** = `min(all impacts)` — the **bottleneck** principle
- The weakest condition determines overall certainty

## This Notebook Demonstrates

1. A rule with **2 weighted conditions** (weights 0.9 and 0.4)
2. How the evidence tree shows the proof structure
3. How certainty propagates through: **condition → impact → bottleneck**
4. The full delivery chain: tree → summary → narrative → NL → audit → static HTML

## 0. Imports

In [1]:
import sys
from pathlib import Path

# Ensure src/ is on the Python path so factpy_kernel is importable.
_src = str(Path(__file__).resolve().parent.parent / "src") if "__file__" in dir() else str(Path.cwd().parent / "src")
if _src not in sys.path:
    sys.path.insert(0, _src)
print("src path:", _src)

src path: /Users/zhenzhili/hnsm-backend/src


In [2]:
from __future__ import annotations

import json
import tempfile
from pathlib import Path
from pprint import pprint

from factpy_kernel.sdk import (
    SDKStore,
    Entity,
    Identity,
    Field,
    Rule,
    Pred,
    vars as sdk_vars,
)
from factpy_kernel.authoring import FileAuthoringRegistry
from factpy_kernel.audit import AuditQuery, load_audit_package
from factpy_kernel.service.runtime_v1 import (
    open_runtime_session,
    close_runtime_session,
    reset_runtime_sessions_for_tests,
    write_runtime_fact,
    evaluate_runtime_derivation,
    explain_runtime_tree,
    explain_runtime_summary,
    explain_runtime_narrative,
    explain_runtime_nl,
    export_runtime_package,
)

## 1. Schema & Rule Definition

A `User` entity with `tag` and `score` fields. The child rule `q.qualified_user`
has **two predicates** in its body, each with a different `condition_weight`:

- `b0.a0` → `user:tag` — weight **0.9** (high importance)
- `b0.a1` → `user:score` — weight **0.4** (lower importance)

This means: even if the tag condition is strong, the weaker score condition
will become the **bottleneck** that limits overall certainty.

In [ ]:
class User(Entity):
    user_id: str = Identity(primary_key=True)
    locale: str = Identity()
    name: str = Field(cardinality="single")
    tag: str = Field(cardinality="multi")
    score: str = Field(cardinality="single")

sdk = SDKStore([User])

# Define a rule with TWO predicates and different condition_weights.
with sdk_vars("u", "tag", "score") as (u, tag, score):
    child_rule = Rule(
        id="q.qualified_user",
        version="1.0.0",
        select=[u, tag],
        where=[
            Pred("user:tag", u, tag),       # b0.a0 — "does user have this tag?"
            Pred("user:score", u, score),   # b0.a1 — "does user have a score?"
        ],
        expose=True,
        condition_weights={
            "b0.a0": 0.9,   # tag condition: high weight
            "b0.a1": 0.4,   # score condition: low weight → will be bottleneck
        },
    )

print("Rule:", child_rule.id)
print("Condition weights:")
print("  b0.a0 (user:tag)  → weight 0.9")
print("  b0.a1 (user:score) → weight 0.4  ← this will be the bottleneck")

## 2. Registry Setup & Seed Data

Register the rule in a temporary registry, then seed a user with a fact.

In [ ]:
registry_dir = tempfile.mkdtemp(prefix="certainty_demo_")
registry = FileAuthoringRegistry(Path(registry_dir))
registry.upsert_schema_ir(sdk.schema_ir)
registry.register_rule_spec(sdk._compile_rule_input(child_rule))

print("Registry root:", registry_dir)
print("Rule spec stored:", registry.read_rule_spec("q.qualified_user", "1.0.0") is not None)

In [ ]:
# Seed a user entity with tag AND score facts.
with sdk.batch() as tx:
    u1 = tx.entity(User, user_id="u-001", locale="en")
    u1.name.set("Alice")
    u1.tag.add("vip")
    u1.score.set("85")
    tx.commit()

u1_ref = sdk.ref(User, user_id="u-001", locale="en")
print("User ref:", u1_ref)
print("Facts: tag='vip', score='85'")

## 3. Runtime Evaluate with Certainty Routing

Open a runtime session **with `registry_root`**. When a native derivation
references a child rule that has `condition_weights`, the
`CertaintyConfidenceKindResolver` automatically marks the candidate as
`confidence_kind="certainty"` at creation time.

In [ ]:
reset_runtime_sessions_for_tests()
session_resp = open_runtime_session({"registry_root": registry_dir})
session_id = session_resp["session"]["session_id"]

# Write both facts into the runtime session's store.
write_runtime_fact(
    session_id,
    {"pred_id": "user:tag", "e_ref": u1_ref, "rest_terms": [["string", "vip"]]},
    kind="add",
)
write_runtime_fact(
    session_id,
    {"pred_id": "user:score", "e_ref": u1_ref, "rest_terms": [["string", "85"]]},
    kind="add",
)

# Evaluate a derivation that references the child rule via ruleref.
eval_resp = evaluate_runtime_derivation(
    session_id,
    {
        "derivation": {
            "derivation_id": "drv.certainty_demo",
            "version": "1.0.0",
            "target": "user:tag",
            "head_vars": ["$u", "$tag"],
            "where": [
                ["ruleref", "q.qualified_user", "1.0.0", ["$u", "$tag"]],
                ["eq", "$tag", "vip"],
            ],
            "mode": "native",
        }
    },
)

candidate = eval_resp["evaluation"]["candidates"][0]
candidate_id = candidate["candidate_id"]

print("Candidate ID:", candidate_id)
print("confidence_kind:", candidate["confidence_kind"])
print()
print(">>> confidence_kind='certainty' — auto-set by resolver because rule has condition_weights")

## 4. Evidence Tree: The Proof Structure

The evidence tree answers: **"Why was this candidate derived?"**

For our example, the proof chain is:
1. The **parent derivation** invoked `q.qualified_user` via `ruleref` → `rule_ref_section`
2. The **child rule** matched with two predicate witnesses → `referenced_support`
3. Each predicate has a **witness group** showing which facts matched → `predicate_witness_group`
4. Each witness group contains the **actual assertion** → `assertion_fact`

This is the subtree where certainty propagation happens:
the `referenced_support` node contains the conditions that get weighted.

In [ ]:
# Fetch the raw evidence tree.
tree_resp = explain_runtime_tree(
    session_id,
    {"kind": "candidate", "id": candidate_id},
)
tree = tree_resp["tree"]

# Top-level metadata.
print("=== Evidence Tree Metadata ===")
print(f"  kind: {tree['kind']}")
print(f"  candidate_id: {tree['candidate_id']}")
print(f"  support_kind: {tree['support_kind']}")
print(f"  support_digest: {tree['support_digest'][:32]}...")
print()

def print_tree(node, indent=0):
    """Pretty-print the evidence tree with node_kind and key fields."""
    prefix = "  " * indent
    kind = node.get("node_kind", "?")
    label = kind

    # Add contextual details per node kind.
    if kind == "candidate_result":
        label += f"  (root_result_kind={node.get('root_result_kind', '?')})"
    elif kind == "support_section":
        label += f"  ({len(node.get('children', []))} children)"
    elif kind == "rule_ref_section":
        label += f"  ({len(node.get('children', []))} refs)"
    elif kind == "predicate_witness_group":
        label += f"  pred_id={node.get('pred_id', '?')}  assertions={node.get('assertion_count', '?')}"
    elif kind == "assertion_fact":
        claims = node.get("claim_args", [])
        terms_str = ", ".join(f"{c.get('val','?')}" for c in claims) if claims else str(node.get("terms", []))
        label += f"  [{terms_str}]"
        conf = node.get("confidence")
        if conf is not None:
            label += f"  confidence={conf}"
    elif kind == "non_fact_check":
        label += f"  {node.get('check_kind', '?')}  status={node.get('status', '?')}"
    elif kind == "rule_ref":
        label += f"  {node.get('rule_ref_id', '?')} v{node.get('rule_ref_version', '?')}"
    elif kind == "referenced_support":
        digest = node.get("support_digest", "?")
        label += f"  digest={digest[:24]}..."

    print(f"{prefix}- {label}")
    for child in node.get("children", []):
        print_tree(child, indent + 1)

# The actual tree nodes start at tree["root"].
print("=== Evidence Tree (hierarchical proof) ===")
print_tree(tree["root"])

### 4b. Tree Summary (12-field deterministic summary)

The tree summary is a flat dict derived from the raw tree. It captures
the structural shape without requiring the full tree traversal.

In [9]:
# The summary endpoint returns both the tree summary and certainty_summary.
# Let's look at the tree summary first (the 12-field structural shape).
summary_resp = explain_runtime_summary(
    session_id,
    {"kind": "candidate", "id": candidate_id},
)

print("=== Tree Summary (structural shape) ===")
tree_summary = summary_resp["summary"]
for key, value in tree_summary.items():
    print(f"  {key}: {value}")

print()
print(">>> This summary is derived purely from the evidence tree above.")

=== Tree Summary (structural shape) ===
  candidate_id: cand_v2:178ca5f85b531f2140c17ae77d4f189167f489454c4c8ac540880ffdacf7e083
  support_kind: native_binding_v1
  is_degraded: False
  root_result_kind: fact
  node_count_by_role: {'structural': 4, 'witness': 2, 'constraint': 2, 'rule_chain': 2, 'terminal': 0, 'degraded': 0}
  witness_assertion_count: 1
  rule_ref_count: 1
  recursive_depth: 1
  has_unresolved: False
  has_boundary: False
  unresolved_reasons: []
  boundary_reasons: []

>>> This summary is derived purely from the evidence tree above.


## 5. Certainty Propagation: From Tree to Summary

Now watch how certainty propagates through the evidence tree:

```
referenced_support (child proof subtree)
├── predicate_witness_group (user:tag)   → atom b0.a0 → weight 0.9 × confidence 1.0 = impact 0.9
└── predicate_witness_group (user:score) → atom b0.a1 → weight 0.4 × confidence 1.0 = impact 0.4  ← BOTTLENECK
                                                                                          ↓
                                                              aggregate_certainty = min(0.9, 0.4) = 0.4
```

The `certainty_summary` captures this computation as structured data:

In [10]:
# certainty_summary is a sibling of the tree summary in the same response.
print("=== Certainty Summary ===")
pprint(summary_resp["certainty_summary"])

=== Certainty Summary ===
{'aggregate_certainty': 0.8,
 'condition_count': 1,
 'conditions': [{'atom_key': 'b0.a0',
                 'impact': 0.8,
                 'node_kind': 'predicate_witness_group',
                 'weight': 0.8}],
 'confidence_kind': 'certainty',
 'weighted_condition_count': 1}


## 6. Narrative: Ranked Conditions + Bottleneck Marker

The narrative layer sorts conditions by impact (ascending — weakest first)
and marks the bottleneck with `[bottleneck]`. It also produces a
machine-readable `certainty_bottleneck` key.

With 2 conditions, you'll see:
- `b0.a1` (score, impact=0.4) listed first as `[bottleneck]`
- `b0.a0` (tag, impact=0.9) listed second

In [11]:
narrative_resp = explain_runtime_narrative(
    session_id,
    {"kind": "candidate", "id": candidate_id},
)
narrative = narrative_resp["narrative"]

print("=== Certainty Lines (ranked by impact) ===")
for line in narrative.get("certainty_lines", []):
    print(" ", line)

print()
print("=== Certainty Bottleneck (machine-readable) ===")
pprint(narrative.get("certainty_bottleneck"))

=== Certainty Lines (ranked by impact) ===
  Certainty (eligible child-proof subtree): aggregate certainty (bottleneck): 0.8.
  Condition b0.a0 (predicate_witness_group): weight=0.8, impact=0.8. [bottleneck]

=== Certainty Bottleneck (machine-readable) ===
{'atom_keys': ['b0.a0'], 'impact': 0.8}


## 7. Explain NL: Weakest-Condition Sentence

The NL layer appends a 5th paragraph with a human-readable summary
of the weakest condition (bottleneck).

In [12]:
nl_resp = explain_runtime_nl(
    session_id,
    {"kind": "candidate", "id": candidate_id},
)

print("=== NL Explain (all paragraphs) ===")
for i, para in enumerate(nl_resp["explain_nl"]["paragraphs"]):
    print(f"  [{i+1}] {para}")
    print()

=== NL Explain (all paragraphs) ===
  [1] Candidate cand_v2:178ca5f85b531f2140c17ae77d4f189167f489454c4c8ac540880ffdacf7e083 uses support kind native_binding_v1 across 10 tree node(s). Root result kind: fact. Role counts: structural=4, witness=2, constraint=2, rule_chain=2, terminal=0, degraded=0. Recursive depth: 1.

  [2] Evidence summary: Witness assertions: 1. Witness nodes: 2; constraint nodes: 2.

  [3] Rule-chain summary: Rule reference nodes: 1. Recursive proof depth: 1.

  [4] Terminal and drill-down summary: No unresolved support or recursion boundaries were encountered. Open referenced support branches to inspect recursive child proof. Open linked assertion nodes to inspect witness facts.

  [5] Certainty summary: Certainty (eligible child-proof subtree): aggregate certainty (bottleneck): 0.8. Condition b0.a0 (predicate_witness_group): weight=0.8, impact=0.8. [bottleneck] The weakest condition is b0.a0 with impact 0.8.



## 8. Audit Round-Trip

Accept the candidate, export an audit package, and verify that both
the evidence tree and certainty summary survive the round-trip.

In [ ]:
from factpy_kernel.service.runtime_v1 import accept_runtime_derivation
from factpy_kernel.audit import render_audit_static_site

accept_resp = accept_runtime_derivation(
    session_id,
    {"candidate": candidate, "options": {"approved_by": "demo"}},
)
print("Accept:", accept_resp["ok"])

# We keep the temp dirs alive so the static site can be displayed below.
import tempfile as _tf
_pkg_dir_obj = _tf.TemporaryDirectory(prefix="certainty_pkg_")
_site_dir_obj = _tf.TemporaryDirectory(prefix="certainty_site_")
pkg_dir = _pkg_dir_obj.name
site_dir = _site_dir_obj.name

export_resp = export_runtime_package(
    session_id,
    {"out_dir": pkg_dir, "package_kind": "audit"},
)
print("Export:", export_resp["ok"])

# Load and query the audit package.
package = load_audit_package(pkg_dir)
audit_query = AuditQuery(package)

# --- Evidence tree round-trip ---
audit_tree = audit_query.get_candidate_evidence_tree(candidate_id)
print()
print("=== Audit Evidence Tree ===")
if audit_tree and "root" in audit_tree:
    print_tree(audit_tree["root"])
elif audit_tree:
    # Fallback: some shapes may not have "root" wrapper.
    print_tree(audit_tree)
else:
    print("  (no tree available)")

audit_tree_summary = audit_query.get_candidate_evidence_tree_summary(candidate_id)
print()
print("=== Audit Tree Summary ===")
if audit_tree_summary:
    for key, value in audit_tree_summary.items():
        print(f"  {key}: {value}")

# --- Certainty round-trip ---
audit_cs = audit_query.get_candidate_certainty_summary(candidate_id)
print()
print("=== Audit Certainty Summary ===")
pprint(audit_cs)

# --- Narrative round-trip (includes certainty_lines) ---
audit_narrative = audit_query.get_candidate_evidence_tree_narrative(candidate_id)
print()
print("=== Audit Certainty Lines ===")
if audit_narrative:
    for line in audit_narrative.get("certainty_lines", []):
        print(" ", line)

# --- Parity checks ---
print()
runtime_cs = summary_resp["certainty_summary"]
print("Certainty parity (runtime == audit):", audit_cs == runtime_cs)
print("Tree summary parity:", audit_tree_summary == tree_summary if audit_tree_summary else "N/A")

# --- Render static site ---
site_result = render_audit_static_site(pkg_dir, site_dir)
print()
print(f"Static site rendered: {site_result.get('pages_written', '?')} pages")
print(f"Site directory: {site_dir}")

### 8b. Static Site: Candidate Evidence HTML Page

The audit static site renders each candidate as a shareable HTML page
with the evidence tree, narrative block, and certainty section.
Below we display the actual generated HTML inline.

In [ ]:
from IPython.display import HTML, display
import glob

# Find the candidate evidence HTML page.
candidate_pages = glob.glob(f"{site_dir}/candidate_evidence/*.html")
if candidate_pages:
    page_path = candidate_pages[0]
    html_content = Path(page_path).read_text(encoding="utf-8")
    print(f"Displaying: {Path(page_path).name}")
    print(f"File size: {len(html_content)} chars")
    print()

    # Display the full HTML page in an iframe for proper rendering.
    display(HTML(f"""
    <div style="border: 2px solid #ccc; border-radius: 8px; overflow: hidden; margin: 10px 0;">
        <div style="background: #f5f5f5; padding: 8px 12px; border-bottom: 1px solid #ccc; font-family: monospace; font-size: 12px;">
            📄 {Path(page_path).name}
        </div>
        <iframe srcdoc='{html_content.replace(chr(39), "&#39;")}' 
                style="width: 100%; height: 600px; border: none;"></iframe>
    </div>
    """))
else:
    print("No candidate evidence pages generated.")
    print(f"Site contents: {list(Path(site_dir).rglob('*.html'))}")

In [ ]:
# List all generated static site pages.
all_pages = sorted(Path(site_dir).rglob("*.html"))
print(f"=== Static Site: {len(all_pages)} HTML pages generated ===")
for p in all_pages:
    rel = p.relative_to(site_dir)
    size = p.stat().st_size
    print(f"  {rel}  ({size} bytes)")

## 9. Negative Case: No `condition_weights` → `confidence_kind="none"`

When a rule does **not** declare `condition_weights`, the resolver
falls back to `"none"` and no certainty summary is produced.

In [ ]:
# Register a rule WITHOUT condition_weights.
with sdk_vars("u", "tag", "score") as (u, tag, score):
    plain_rule = Rule(
        id="q.plain_rule",
        version="1.0.0",
        select=[u, tag],
        where=[
            Pred("user:tag", u, tag),
            Pred("user:score", u, score),
        ],
        expose=True,
        # No condition_weights!
    )
registry.register_rule_spec(sdk._compile_rule_input(plain_rule))

eval_resp2 = evaluate_runtime_derivation(
    session_id,
    {
        "derivation": {
            "derivation_id": "drv.plain_demo",
            "version": "1.0.0",
            "target": "user:tag",
            "head_vars": ["$u", "$tag"],
            "where": [
                ["ruleref", "q.plain_rule", "1.0.0", ["$u", "$tag"]],
                ["eq", "$tag", "vip"],
            ],
            "mode": "native",
        }
    },
)

plain_candidate = eval_resp2["evaluation"]["candidates"][0]
print("Plain candidate confidence_kind:", plain_candidate["confidence_kind"])

plain_summary = explain_runtime_summary(
    session_id,
    {"kind": "candidate", "id": plain_candidate["candidate_id"]},
)
print("Certainty summary:", plain_summary.get("certainty_summary"))
print()
print(">>> Same rule structure, but no condition_weights")
print(">>> → resolver falls back to 'none' → no certainty delivery")

In [ ]:
# Cleanup
close_runtime_session(session_id)
reset_runtime_sessions_for_tests()
_pkg_dir_obj.cleanup()
_site_dir_obj.cleanup()
print("Session closed, temp dirs cleaned.")

## Summary

| Step | What happens |
|------|--------------|
| Rule declaration | `condition_weights={"b0.a0": 0.8}` on child rule |
| Registry | Rule payload stored with weights |
| Evaluate | `CertaintyConfidenceKindResolver` checks eligibility → `confidence_kind="certainty"` |
| **Evidence tree** | **Raw hierarchical proof: `candidate_result → support_section → predicate_witness_group → assertion_fact`** |
| **Tree summary** | **12-field deterministic summary derived from tree** |
| Certainty summary | `certainty_summary` with per-condition `impact` and `aggregate_certainty` |
| Narrative | `certainty_lines` sorted by impact, `[bottleneck]` marked |
| NL | 5th paragraph with weakest-condition sentence |
| Audit export | `certainty_summaries.jsonl` + `support_artifacts.jsonl` materialized at export time |
| Audit query | Evidence tree + certainty summary + narrative all survive round-trip |
| **Static HTML** | **`render_audit_static_site` generates candidate evidence pages with certainty section** |
| No weights | Falls back to `confidence_kind="none"`, no certainty delivery |